# DIT5411 — Full Pipeline + Fine-tune (All-in-One)

**Student:** Chong Ka Ho  
**Framework:** TensorFlow / Keras  
**Classes:** 13,065  

This notebook includes:
1. Full data pipeline (download → split → arrays)
2. Augmentation
3. Model definitions
4. Recorded baseline results
5. **Fine-tune MobileNetV2** (try for higher score)

### Recorded baseline (already obtained)
| Model | Test Acc |
| --- | ---: |
| Model_1_MLP | 0.0001 |
| Model_2_Scratch_CNN | 0.0001 |
| Model_3_MobileNetV2 (frozen) | 0.0578 |
| Diagnostic 100-class CNN | 0.4830 |

### How to run
1. `Runtime → Change runtime type → GPU`
2. Run **Part 1 → 7** in order
3. Part 8 is the fine-tune section (needs GPU; ~2–4 hours)

GPU check must NOT print `[]`.


## Part 1 — Setup


In [1]:
import os, re, gc, json, zipfile, subprocess
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm.auto import tqdm

print("TensorFlow", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("GPU:", tf.config.list_physical_devices("GPU"))

NUM_CLASSES = 13065
IMG_SIZE = 64
TRAIN_ORIGINALS = 40
TARGET_TRAIN = 200
BATCH_SIZE = 128
SEED = 42

WORK = Path("/content/dit5411")
WORK.mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK / "handwritting_data_all"
PREPARED = WORK / "prepared"
OUTPUTS = WORK / "outputs"
PREPARED.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)


TensorFlow 2.21.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Part 2 — Download full dataset (13,065 characters)


In [2]:
if not (DATA_ROOT / "all_data.zip").exists() and not list(DATA_ROOT.glob("all_data.zip.*")):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/chenkenanalytic/handwritting_data_all.git",
        str(DATA_ROOT),
    ])

zip_path = DATA_ROOT / "all_data.zip"
parts = sorted(DATA_ROOT.glob("all_data.zip.*"))
if not zip_path.exists() and parts:
    print("Concatenating split zip parts...")
    with open(zip_path, "wb") as out:
        for part in parts:
            print(" ", part.name)
            out.write(part.read_bytes())

assert zip_path.exists(), "all_data.zip not found"
print("Zip size GB:", round(zip_path.stat().st_size / 1e9, 2))


Concatenating split zip parts...
  all_data.zip.001
  all_data.zip.002
  all_data.zip.003
  all_data.zip.004
  all_data.zip.005
  all_data.zip.006
  all_data.zip.007
  all_data.zip.008
  all_data.zip.009
  all_data.zip.010
  all_data.zip.011
  all_data.zip.012
Zip size GB: 1.14


## Part 3 — Build arrays (first 40 train / remaining test)


In [3]:
def decode_zip_name(raw_name: str) -> str:
    try:
        return raw_name.encode("cp437").decode("big5")
    except UnicodeError:
        return raw_name

def sample_index(name: str) -> int:
    stem = Path(name).stem
    if "_" not in stem:
        return 0
    m = re.match(r"(\d+)", stem.split("_", 1)[1])
    return int(m.group(1)) if m else 0

arrays_path = PREPARED / "arrays.npz"

if arrays_path.exists():
    print("Found existing arrays.npz, skipping rebuild")
else:
    grouped = defaultdict(list)
    label_map = {}
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if (not i.is_dir()) and i.filename.lower().endswith(".png")]
        for info in tqdm(infos, desc="Indexing zip"):
            name = decode_zip_name(info.filename)
            parts_ = name.replace("\\", "/").split("/")
            if len(parts_) < 3:
                continue
            try:
                folder_id = int(parts_[1])
            except ValueError:
                continue
            class_id = folder_id - 1
            if not (0 <= class_id < NUM_CLASSES):
                continue
            label_map[f"{class_id:05d}"] = Path(parts_[-1]).stem.split("_", 1)[0]
            grouped[class_id].append((sample_index(name), zf.read(info)))

    assert len(grouped) == NUM_CLASSES, f"expected {NUM_CLASSES}, got {len(grouped)}"

    train_images, train_labels = [], []
    test_images, test_labels = [], []
    per_class_train = np.zeros(NUM_CLASSES, dtype=np.int32)
    per_class_test = np.zeros(NUM_CLASSES, dtype=np.int32)

    for class_id in tqdm(range(NUM_CLASSES), desc="Decode/resize"):
        payloads = [p for _, p in sorted(grouped[class_id], key=lambda x: x[0])]
        if len(payloads) > TRAIN_ORIGINALS:
            n_train = TRAIN_ORIGINALS
        elif len(payloads) > 1:
            n_train = len(payloads) - 1
        else:
            n_train = len(payloads)

        for raw in payloads[:n_train]:
            im = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
            if im is None:
                continue
            im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            train_images.append(im)
            train_labels.append(class_id)
            per_class_train[class_id] += 1

        for raw in payloads[n_train:]:
            im = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
            if im is None:
                continue
            im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            test_images.append(im)
            test_labels.append(class_id)
            per_class_test[class_id] += 1

        if per_class_train[class_id] == 0:
            raise RuntimeError(f"No train images for class {class_id}")

    X_train = np.stack(train_images)[..., None].astype(np.uint8)
    y_train = np.asarray(train_labels, np.int32)
    X_test = np.stack(test_images)[..., None].astype(np.uint8)
    y_test = np.asarray(test_labels, np.int32)

    np.savez_compressed(
        arrays_path,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        per_class_train=per_class_train, per_class_test=per_class_test,
    )
    meta = {
        "num_classes": NUM_CLASSES,
        "img_size": IMG_SIZE,
        "X_train_shape": list(X_train.shape),
        "X_test_shape": list(X_test.shape),
        "classes_without_test": int(np.sum(per_class_test == 0)),
    }
    (PREPARED / "meta.json").write_text(json.dumps(meta, indent=2))
    (PREPARED / "label_map.json").write_text(json.dumps(label_map, ensure_ascii=False, indent=2))
    print(meta)
    del grouped, train_images, test_images
    gc.collect()

data = np.load(arrays_path)
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
print("X_train", X_train.shape, "X_test", X_test.shape)
assert len(np.unique(y_train)) == NUM_CLASSES

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(y_train))
X_train, y_train = X_train[perm], y_train[perm]
n_val = max(1, int(len(y_train) * 0.1))
X_val, y_val = X_train[:n_val], y_train[:n_val]
X_fit, y_fit = X_train[n_val:], y_train[n_val:]
input_shape = X_train.shape[1:]
print("fit/val/test:", len(y_fit), len(y_val), len(y_test))
del data
gc.collect()


Indexing zip:   0%|          | 0/684677 [00:00<?, ?it/s]

Decode/resize:   0%|          | 0/13065 [00:00<?, ?it/s]

{'num_classes': 13065, 'img_size': 64, 'X_train_shape': [522084, 64, 64, 1], 'X_test_shape': [162593, 64, 64, 1], 'classes_without_test': 0}
X_train (522084, 64, 64, 1) X_test (162593, 64, 64, 1)
fit/val/test: 469876 52208 162593


44

## Part 4 — Augmentation method

- `RandomRotation` ≈ rotation
- `RandomZoom` ≈ scaling
- `RandomTranslation` ≈ shear-like distortion

Across epochs with live augmentation, each character gets **≥ 200** augmented views.


In [4]:
STEPS = max(1, len(X_fit) // BATCH_SIZE)
print("steps_per_epoch =", STEPS)


steps_per_epoch = 3670


## Part 5 — Model definitions (output = 13,065)


In [5]:
def build_mlp_model(input_shape, num_classes):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1 / 255.0),
        layers.Flatten(),
        layers.Dense(1024, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ], name="Model_1_MLP")

def build_cnn_model(input_shape, num_classes):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1 / 255.0),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ], name="Model_2_Scratch_CNN")

def build_mobilenet_model(num_classes, trainable_base=False):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255.0)(inp)
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = trainable_base
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inp, out, name="Model_3_MobileNetV2")
    return model, base

print("Models ready. Output classes =", NUM_CLASSES)


Models ready. Output classes = 13065


## Part 6 — Recorded baseline results (no long re-train needed)


In [6]:
results = {
    "Model_1_MLP": {
        "test_acc": 0.0001,
        "test_loss": 9.48,
        "final_train_acc": 0.0,
        "final_val_acc": 0.0,
        "epochs_ran": 2,
        "description": "MLP baseline",
    },
    "Model_2_Scratch_CNN": {
        "test_acc": 0.0001,
        "test_loss": 9.48,
        "final_train_acc": 0.0,
        "final_val_acc": 0.0,
        "epochs_ran": 4,
        "description": "From-scratch CNN short runs",
    },
    "Model_3_MobileNetV2": {
        "test_acc": 0.0578,
        "test_loss": 6.42,
        "final_train_acc": 0.0998,
        "final_val_acc": 0.0603,
        "epochs_ran": 8,
        "description": "Frozen MobileNetV2 + Dense(13065)",
    },
}

diagnostic = {
    "100_class_scratch_cnn_test_acc": 0.4830,
    "note": "Same pipeline on 100 classes reached 48.3%, proving data/labels are correct.",
}

best = max(results, key=lambda k: results[k]["test_acc"])
summary = {
    "best_model": best,
    "best_test_acc": results[best]["test_acc"],
    "results": results,
    "diagnostic": diagnostic,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "augmentation": ["RandomRotation", "RandomZoom", "RandomTranslation"],
}
(OUTPUTS / "results.json").write_text(json.dumps(summary, indent=2))

print("===== Baseline Comparison =====")
for name, r in results.items():
    print(f"{name}: Test={r['test_acc']:.4f}")
print("Best so far:", best, results[best]["test_acc"])
print("Saved:", OUTPUTS / "results.json")


===== Baseline Comparison =====
Model_1_MLP: Test=0.0001
Model_2_Scratch_CNN: Test=0.0001
Model_3_MobileNetV2: Test=0.0578
Best so far: Model_3_MobileNetV2 0.0578
Saved: /content/dit5411/outputs/results.json


## Part 7 — References


1. https://github.com/AI-FREE-Team/Traditional-Chinese-Handwriting-Dataset  
2. https://github.com/chenkenanalytic/handwritting_data_all  
3. https://github.com/darshitajain/Introduction-to-OpenCV  
4. https://www.tensorflow.org/  
5. https://github.com/kimanalytics/Handwritten-Digit-Recognition-using-Keras-and-TensorFlow


## Part 8 — Fine-tune MobileNetV2 (try for higher score)

**Goal:** beat previous best **0.0578**

- Phase 1: train head only (frozen backbone)
- Phase 2: unfreeze last ~30 layers, small LR
- Needs GPU; expect about **2–4 hours** total

If GPU prints `[]`, stop and enable GPU first.


### Part 8.1 — RGB datasets + light aug


In [7]:
assert tf.config.list_physical_devices("GPU"), "No GPU detected. Enable GPU before fine-tuning."

def gray_to_rgb(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.repeat(images, repeats=3, axis=-1)
    return images, labels

aug = tf.keras.Sequential([
    layers.RandomRotation(0.05, fill_mode="constant", fill_value=255.0),
    layers.RandomZoom((-0.1, 0.1), fill_mode="constant", fill_value=255.0),
    layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=255.0),
], name="light_aug")

def make_rgb_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(8000, seed=SEED, reshuffle_each_iteration=True).repeat()
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(gray_to_rgb, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        def _aug(images, labels):
            images = aug(images, training=True)
            images = tf.clip_by_value(images, 0.0, 255.0)
            return images, labels
        ds = ds.map(_aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_rgb_ds(X_fit, y_fit, training=True)
val_ds = make_rgb_ds(X_val, y_val, training=False)
test_ds = make_rgb_ds(X_test, y_test, training=False)
print("Fine-tune datasets ready. steps_per_epoch =", STEPS)


Fine-tune datasets ready. steps_per_epoch = 3670


### Part 8.2 — Phase 1: frozen backbone


In [8]:
tf.keras.backend.clear_session()
gc.collect()

model, base = build_mobilenet_model(NUM_CLASSES, trainable_base=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history1 = model.fit(
    train_ds,
    epochs=6,
    steps_per_epoch=STEPS,
    validation_data=val_ds,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5
        ),
    ],
    verbose=1,
)

phase1_loss, phase1_acc = model.evaluate(test_ds, verbose=0)
print("Phase 1 TEST ACC =", round(phase1_acc, 4))


/tmp/ipykernel_363/1581311852.py:32: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 420s 107ms/step - accuracy: 0.0075 - loss: 10.7303 - val_accuracy: 0.0104 - val_loss: 9.6912 - learning_rate: 0.0010
Epoch 2/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 367s 100ms/step - accuracy: 0.0205 - loss: 9.1536 - val_accuracy: 0.0157 - val_loss: 9.3677 - learning_rate: 0.0010
Epoch 3/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 367s 100ms/step - accuracy: 0.0310 - loss: 8.6106 - val_accuracy: 0.0199 - val_loss: 9.2906 - learning_rate: 0.0010
Epoch 4/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 366s 100ms/step - accuracy: 0.0391 - loss: 8.3298 - val_accuracy: 0.0226 - val_loss: 9.3588 - learning_rate: 0.0010
Epoch 5/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 363s 99ms/step - accuracy: 0.0540 - loss: 7.5675 - val_accuracy: 0.0279 - val_loss: 8.8952 - learning_rate: 5.0000e-04
Epoch 6/6
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 358s 98ms/step - accuracy: 0.0598 - loss: 7.3488 - val_accuracy: 0.0299 - val_loss: 8.8831 - learning_rate: 5.0000e-04
Ph

### Part 8.3 — Phase 2: fine-tune top layers


In [9]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

print("Trainable base layers:", sum(1 for l in base.layers if l.trainable))

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history2 = model.fit(
    train_ds,
    epochs=8,
    steps_per_epoch=STEPS,
    validation_data=val_ds,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(OUTPUTS / "MobileNetV2_Finetune.keras"),
            monitor="val_loss",
            save_best_only=True,
        ),
    ],
    verbose=1,
)

phase2_loss, phase2_acc = model.evaluate(test_ds, verbose=0)
print("Phase 2 TEST ACC =", round(phase2_acc, 4))
print("Previous best = 0.0578")
print("Improved?", phase2_acc > 0.0578)


Trainable base layers: 30
Epoch 1/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 396s 102ms/step - accuracy: 0.0922 - loss: 6.0190 - val_accuracy: 0.1407 - val_loss: 4.7869 - learning_rate: 1.0000e-04
Epoch 2/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 370s 101ms/step - accuracy: 0.2822 - loss: 3.4195 - val_accuracy: 0.2312 - val_loss: 4.0625 - learning_rate: 1.0000e-04
Epoch 3/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 369s 100ms/step - accuracy: 0.3883 - loss: 2.6889 - val_accuracy: 0.3119 - val_loss: 3.3386 - learning_rate: 1.0000e-04
Epoch 4/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 375s 102ms/step - accuracy: 0.4596 - loss: 2.2673 - val_accuracy: 0.3431 - val_loss: 3.0921 - learning_rate: 1.0000e-04
Epoch 5/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 377s 103ms/step - accuracy: 0.5133 - loss: 1.9805 - val_accuracy: 0.4329 - val_loss: 2.4683 - learning_rate: 1.0000e-04
Epoch 6/8
3670/3670 ━━━━━━━━━━━━━━━━━━━━ 367s 100ms/step - accuracy: 0.5552 - loss: 1.7665 - val_accuracy: 0.4545 - val_loss: 2.3106 - learning_rate: 1.0000e-04
Epoch 7/

### Part 8.4 — Save fine-tune results


In [10]:
results_ft = {
    "Model_1_MLP": {"test_acc": 0.0001},
    "Model_2_Scratch_CNN": {"test_acc": 0.0001},
    "Model_3_MobileNetV2_frozen_baseline": {"test_acc": 0.0578},
    "Model_3_MobileNetV2_phase1": {
        "test_acc": float(phase1_acc),
        "test_loss": float(phase1_loss),
    },
    "Model_3_MobileNetV2_finetuned": {
        "test_acc": float(phase2_acc),
        "test_loss": float(phase2_loss),
    },
}

best_ft = max(results_ft, key=lambda k: results_ft[k]["test_acc"])
summary_ft = {
    "best_model": best_ft,
    "best_test_acc": results_ft[best_ft]["test_acc"],
    "previous_best_test_acc": 0.0578,
    "results": results_ft,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "steps_per_epoch": STEPS,
    "augmentation": ["RandomRotation", "RandomZoom", "RandomTranslation"],
}
(OUTPUTS / "results_finetune.json").write_text(json.dumps(summary_ft, indent=2))

print("===== Fine-tune Comparison =====")
for k, v in results_ft.items():
    print(f"{k}: Test={v['test_acc']:.4f}")
print("Best:", best_ft, results_ft[best_ft]["test_acc"])
print("Saved:", OUTPUTS / "results_finetune.json")

# Optional Drive backup
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(OUTPUTS / "results_finetune.json", "/content/drive/MyDrive/results_finetune.json")


===== Fine-tune Comparison =====
Model_1_MLP: Test=0.0001
Model_2_Scratch_CNN: Test=0.0001
Model_3_MobileNetV2_frozen_baseline: Test=0.0578
Model_3_MobileNetV2_phase1: Test=0.0315
Model_3_MobileNetV2_finetuned: Test=0.4902
Best: Model_3_MobileNetV2_finetuned 0.4902486503124237
Saved: /content/dit5411/outputs/results_finetune.json
